In [10]:
import numpy as np
import pandas as pd
movies = pd.read_csv(r"C:\Users\amarn\OneDrive\Pictures\New folder\tmdb_5000_movies.csv")
credits = pd.read_csv(r"C:\Users\amarn\OneDrive\Pictures\New folder\tmdb_5000_credits.csv")
movies = movies.merge(credits,left_on ='id',right_on = 'movie_id')
movies = movies[['id','title_x','overview','genres','keywords','cast','crew']]
movies.dropna(inplace =True)
movies.rename(columns ={'title_x':'title'},inplace = True)
movies.drop_duplicates(subset=['id'], inplace=True)


In [11]:
import ast

def convert(text):
    if isinstance(text, str):
        return [i['name'] for i in ast.literal_eval(text)]
    else:
        return text
def convert_crew(text):
    l = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            l.append(i['name'])
            break
    return l
movies['crew'] = movies['crew'].apply(convert_crew)
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)



In [12]:
def clean(text):
    return [i.replace(" ", "").lower() for i in text]

movies['genres'] = movies['genres'].apply(clean)
movies['keywords'] = movies['keywords'].apply(clean)
movies['crew'] = movies['crew'].apply(clean)

In [13]:
movies['genres']   = movies['genres'].apply(lambda x: ' '.join(x))
movies['keywords'] = movies['keywords'].apply(lambda x: ' '.join(x))
movies['cast']     = movies['cast'].apply(lambda x: ' '.join(x))
movies['crew']     = movies['crew'].apply(lambda x: ' '.join(x))
movies['overview'] = movies['overview'].apply(lambda x: x.lower())
movies['tags'] = (
    movies['overview']
    + ' ' + movies['genres'].apply(lambda x: ' '.join(x))
    + ' ' + movies['keywords'].apply(lambda x: ' '.join(x))
    + ' ' + movies['cast'].apply(lambda x: ' '.join(x))
    + ' ' + movies['crew'].apply(lambda x: ' '.join(x))
)




In [14]:
new_df = movies[['id', 'title', 'tags']]

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors.shape


(4800, 5000)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [17]:
def recommend(movie):
    movie = movie.lower()
    index = new_df[new_df['title'].str.lower() ==movie].index[0]
    distances = similarity[index]
    movie_list =sorted(list(enumerate(distances)),reverse =True,key = lambda x:x[1])[1:6]
    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [18]:
import pickle
pickle.dump(new_df,open('movie.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))